# EDA INDICADORES RENTA - Análisis de datos económicos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

### Carga y exploración de estructura

In [ ]:
df = pd.read_parquet("../data/processed/01_consolidated/dataset_renta.parquet")

print(f"\n ESTRUCTURA RENTA")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Tamaño memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.head()



 ESTRUCTURA RENTA
  Filas: 143,262
  Columnas: 6
  Tamaño memoria: 22.50 MB


,Municipios,Distritos,Secciones,Indicadores de renta media,Periodo,Total
0,28073 Humanes de Madrid,NaN,NaN,Renta neta media por persona,2023,12.552
1,28073 Humanes de Madrid,NaN,NaN,Renta neta media por persona,2022,11.744
2,28073 Humanes de Madrid,NaN,NaN,Renta neta media por persona,2021,10.929
3,28073 Humanes de Madrid,NaN,NaN,Renta neta media por persona,2020,10.298
4,28073 Humanes de Madrid,NaN,NaN,Renta neta media por persona,2019,10.211


### Análisis de nulos

In [25]:

nulos = df.isnull().sum()   
if nulos.sum() == 0:
    print(f"\n Sin valores nulos")
    
print(f"\n  Valores nulos por columna:")
nulos_df = pd.DataFrame({
        'Columna': nulos.index,
        'Nulos': nulos.values,
        '%': (nulos.values / len(df) * 100).round(2)
    })
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
for _, row in nulos_df.iterrows():
        print(f"  {row['Columna']:30s} {row['Nulos']:>8,d}  ({row['%']:>6.2f}%)")


  Valores nulos por columna:
  Total                             3,418  (  2.39%)
  Secciones                         1,944  (  1.36%)
  Distritos                           378  (  0.26%)


### Análisis de duplicados

In [26]:
duplicados_totales = df.duplicated().sum()
print(f"  Total (todas las columnas): {duplicados_totales:,}")

  Total (todas las columnas): 0


### Análisis de tipos de datos

In [27]:
print(f"\n Tipos de datos:")
tipos = df.dtypes.value_counts()
for tipo, cantidad in tipos.items():
    print(f"  {str(tipo):25s} {cantidad:3d} columnas")

df.info()


 Tipos de datos:
  str                         5 columnas
  int64                       1 columnas
<class 'pandas.DataFrame'>
RangeIndex: 143262 entries, 0 to 143261
Data columns (total 6 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   Municipios                  143262 non-null  str  
 1   Distritos                   142884 non-null  str  
 2   Secciones                   141318 non-null  str  
 3   Indicadores de renta media  143262 non-null  str  
 4   Periodo                     143262 non-null  int64
 5   Total                       139844 non-null  str  
dtypes: int64(1), str(5)
memory usage: 21.5 MB


### Análisis de cobertura temporal

In [29]:
print(f"\n COBERTURA TEMPORAL:")
    
if 'Periodo' in df.columns:
    anios = sorted(df['Periodo'].unique())
    print(f"  Años disponibles: {anios}")
    print(f"  Rango: {anios[0]} → {anios[-1]} ({len(anios)} años)")
        
    # Gap después de 2023
    max_anio = max(anios)
    if max_anio < 2026:
        gap_anios = 2026 - max_anio
        print(f"      GAP: Sin datos desde {max_anio+1} ({gap_anios} años)")
        print(f"      SOLUCIÓN: Forward fill con valor {max_anio}")



 COBERTURA TEMPORAL:
  Años disponibles: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
  Rango: 2015 → 2023 (9 años)
      GAP: Sin datos desde 2024 (3 años)
      SOLUCIÓN: Forward fill con valor 2023


### Análisis de indicadores de disponibilidad

In [30]:
print(f"\n INDICADORES DISPONIBLES:")
    
if 'Indicadores de renta media' in df.columns:
    indicadores = df['Indicadores de renta media'].unique()
    print(f"  Total: {len(indicadores)}")
        
    for i, ind in enumerate(indicadores, 1):
        count = len(df[df['Indicadores de renta media'] == ind])
        print(f"  {i}. {ind} ({count:,} registros)")


 INDICADORES DISPONIBLES:
  Total: 6
  1. Renta neta media por persona (23,877 registros)
  2. Renta neta media por hogar (23,877 registros)
  3. Media de la renta por unidad de consumo (23,877 registros)
  4. Mediana de la renta por unidad de consumo (23,877 registros)
  5. Renta bruta media por persona (23,877 registros)
  6. Renta bruta media por hogar (23,877 registros)


### Análisis de cobertura geográfica

In [31]:
print(f"\n  COBERTURA GEOGRÁFICA:")
    
if 'Distritos' in df.columns:
    distritos = df['Distritos'].nunique()
    print(f"  Distritos: {distritos}")
    
if 'Municipios' in df.columns:
    municipios = df['Municipios'].nunique()
    print(f"  Municipios: {municipios}")
        
    # Solo Madrid?
    if 'Madrid' in df['Municipios'].values:
        madrid_count = len(df[df['Municipios'].str.contains('Madrid', na=False)])
        print(f"  Registros Madrid: {madrid_count:,}")
    
if 'Secciones' in df.columns:
    secciones = df['Secciones'].nunique()
    print(f"  Secciones: {secciones}")


  COBERTURA GEOGRÁFICA:
  Distritos: 29
  Municipios: 7
  Secciones: 2617


### Estadísticas descriptivas

In [ ]:
print(f"\n ESTADÍSTICAS NUMÉRICAS:")
    
numericas = df.select_dtypes(include=[np.number]).columns
if len(numericas) == 0:
    print("  No hay columnas numéricas")
    
    
for col in numericas[:5]:  
    print(f"  {col}:")
    print(f"    Media: {df[col].mean():>12,.0f}")
    print(f"    Min: {df[col].min():>13,.0f}, Max: {df[col].max():>12,.0f}")


 ESTADÍSTICAS NUMÉRICAS:
  Periodo:
    Media:        2,019
    Min:         2,015, Max:        2,023


### Análisis de completitud de datos

In [33]:
print(f"\n  COMPLETITUD:")
    
# Por año e indicador
if 'Periodo' in df.columns and 'Indicadores de renta media' in df.columns:
    anios = sorted(df['Periodo'].unique())
    indicadores = df['Indicadores de renta media'].unique()
        
    print(f"  Matrix: {len(anios)} años × {len(indicadores)} indicadores")
    print(f"  Esperados: {len(anios) * len(indicadores)} registros")
    print(f"  Reales: {len(df):,} registros")
        
    completitud = len(df) / (len(anios) * len(indicadores)) * 100
    print(f"  Completitud: {completitud:.1f}%")



  COMPLETITUD:
  Matrix: 9 años × 6 indicadores
  Esperados: 54 registros
  Reales: 143,262 registros
  Completitud: 265300.0%
